In [1]:
#Define a tabela e o perfil
#perfil = "SRD1CIV"
#perfil = "CAN1CIV"
#perfil = "GDO1CIV"
perfil = "STM1CIV"
#perfil = "SRO2CR"
#perfil = "POA08FZFC"



In [2]:
# Importa tudo

import sqlite3
import pandas as pd

#Bibliotecas de Sistema
from datetime import datetime
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false
from pydoc import text
from sympy import true

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

db = "urcaciv.db"

In [6]:
# Gera um arquivo Excel com os resumos dos pedidos

conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()
table = perfil

cursor.execute(f"SELECT num_processo, tipo, dias, resumo, dt_inclusao, pet FROM {table} WHERE resumo IS NOT NULL")
result = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]
df_resumo = pd.DataFrame(result, columns=columns)
df_resumo.insert(0, "unidade", table)
pos = df_resumo.columns.get_loc("dias") + 1
df_resumo.insert(pos, "tipo de pedido", pd.NA)
# manter apenas a data (dia/mês/ano) na coluna dt_inclusao
df_resumo['dt_inclusao'] = pd.to_datetime(df_resumo['dt_inclusao'], dayfirst=True, errors='coerce').dt.strftime('%d/%m/%Y')
df_resumo.to_excel(f"{table}.xlsx", index=False)

conn.close()


In [7]:
df_resumo

,unidade,num_processo,tipo,dias,tipo de pedido,resumo,dt_inclusao,pet
0,STM1CIV,5046124-73.2024.8.21.0027,EXECUÇÃO FISCAL,0,<NA>,Solicita-se a atualização do endereço da parte...,09/09/2025,ESTADO DO RIO GRANDE DO \nSUL \nPREFEITURA MUN...
1,STM1CIV,5046123-88.2024.8.21.0027,EXECUÇÃO FISCAL,70,<NA>,"Texto maior do que 10.000 bytes, provavelmente...",03/07/2025,EXMO. SR. DR. JUIZ DE DIREITO DA 1ª VARA CÍVEL...
2,STM1CIV,5045971-40.2024.8.21.0027,EXECUÇÃO FISCAL,43,<NA>,A procuradoria jurídica solicita a suspensão d...,29/07/2025,\n \n \nAO\n \nDOUTO\n \nJUI\ń\nZO\n \nDA\n...
3,STM1CIV,5045958-41.2024.8.21.0027,EXECUÇÃO FISCAL,62,<NA>,O Município de Santa Maria requere que o proce...,09/07/2025,\n \n \n \n \n \nEXCELENTISSIMO SENHOR DOUT...
4,STM1CIV,5045691-69.2024.8.21.0027,EXECUÇÃO FISCAL,47,<NA>,"Texto maior do que 10.000 bytes, provavelmente...",24/07/2025,ESTADO DO RIO GRANDE DO \nSUL \nPREFEITURA MUN...
...,...,...,...,...,...,...,...,...
2825,STM1CIV,5000093-54.2008.8.21.0027,CUMPRIMENTO DE SENTENÇA,61,<NA>,O Departamento de Trânsito do Rio Grande do Su...,17/09/2025,ESTADO DO RIO GRANDE DO SUL\nPROCURADORIA-GERA...
2826,STM1CIV,5000088-27.2011.8.21.0027,EXECUÇÃO DE TÍTULO EXTRAJUDICIAL,40,<NA>,O Município solicita a expedição de um ofício ...,09/10/2025,\n \n \n \n 1 EXCELENTÍSSIMO(A) SENHOR(A) DOU...
2827,STM1CIV,5000053-09.2007.8.21.0027,EXECUÇÃO FISCAL,13,<NA>,O Município de Santa Maria requer o prosseguim...,05/11/2025,"\n \n \nRua Venâncio Aires, 2277, 6º andar · ..."
2828,STM1CIV,5000051-05.2008.8.21.0027,EXECUÇÃO FISCAL,19,<NA>,O município solicita a inclusão dos herdeiros ...,31/10/2025,\n \n \nEXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE...


In [5]:
# Mostra os tipos de pedidos conhecidos'
with sqlite3.connect(f"{db}") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT id, resumo FROM tipos_pedidos")
    tipos_pedidos = cursor.fetchall()
    tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])
    print(tipos_pedidos)

ID: 1, Resumo: Pedido de desistência do processo
ID: 2, Resumo: Informação de que a parte está ciente.
ID: 3, Resumo: Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança
ID: 4, Resumo: solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução de baixo valor
ID: 5, Resumo: Solicita a desconsideração do valor irrisório bloqueado e regular prosseguimento do processo
ID: 6, Resumo: Solicita a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas
ID: 7, Resumo: 


In [6]:
# Apaga da tabela PERFIL por id

idt = input("Digite o ID do registro que deseja remover: ")
# Remover o registro da tabela "perfil" onde o id for igual ao fornecido

# Remover os campos "pet" e "resumo" da tabela "perfil" onde o id for 100

conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

cursor.execute(f"""
    UPDATE {table}
    SET pet = NULL, resumo = NULL
    WHERE id = "{idt}"
""")

conn.commit()
conn.close()

In [7]:
# Insere um novo item na tabela tipos_pedidos usando as variáveis id e resumo

resumo = input("Digite o resumo do novo tipo de pedido: ")  # Solicita apenas o resumo ao usuário

with sqlite3.connect(f"{db}") as conn:
    cursor = conn.cursor()
    cursor.execute("INSERT INTO tipos_pedidos (resumo) VALUES (?)", (resumo,))
    id_novo = cursor.lastrowid
    conn.commit()
    print(f"Item inserido: ID: {id_novo}, Resumo: {resumo}")


Item inserido: ID: 8, Resumo: 


In [8]:
# Apaga da tabela TIPOS por id

idt = input("Digite o ID do registro que deseja remover: ")
# Remover o registro da tabela "perfil" onde o id for igual ao fornecido

# Remover os campos "pet" e "resumo" da tabela "perfil" onde o id for 100

conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

cursor.execute(f"""
    UPDATE {table}
    SET pet = NULL, resumo = NULL
    WHERE id = {idt}
""")

conn.commit()
conn.close()

OperationalError: incomplete input

In [ ]:
# Apaga "pet" se o valor anterior não-nulo for idêntico

conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

# Busca todos os registros ordenados por id
cursor.execute(f"SELECT id, pet FROM {table} ORDER BY id")
rows = cursor.fetchall()

prev_pet = None
for row in rows:
    current_id, current_pet = row
    if current_pet is not None:
        if prev_pet == current_pet:
            cursor.execute(f"UPDATE {table} SET pet = NULL WHERE id = ?", (current_id,))
        prev_pet = current_pet

conn.commit()
conn.close()

In [ ]:
# Atualiza o campo "lote" para cada registro com resumo, usando verifica_tipos_de_pedidos


def verifica_tipos_de_pedidos(pedido, lista_de_pedidos):
    print("========== Verificando se é um caso de uso conhecido... ==========")

    with sqlite3.connect(f"{db}") as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT id, resumo FROM tipos_pedidos")
        tipos_pedidos = cursor.fetchall()
        tipos_pedidos = "\n".join([f"Número Identificafdor: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])        

    pergunta_gemma = "Considere a seguinte lista de pedidos:" \
    f"\n{lista_de_pedidos}" \
    f"\nÉ possível dizer que o pedido informado pode ser adequadamente descrito por um item dessa lista?." \
    f"\nRetorne APENAS o Número Identificador do item correspondente na lista, e mais nada." \
    f"\nPedido:" \
    f"\n{pedido}"

    print(f"{pergunta_gemma}")

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])

cursor.execute(f"SELECT id, resumo, pet FROM {table} WHERE resumo IS NOT NULL AND lote IS NULL")
rows = cursor.fetchall()

for row in rows:
    registro_id, resumo, pet = row
    print(f"Processando registro ID {registro_id}")
    #print(f"Pedido:") 
    #print(f"{pet}")    
    resultado = verifica_tipos_de_pedidos(pet, tipos_pedidos)
    print(f"Resultado da verificação: {resultado}")
    if resultado and len(resultado.strip()) > 2:
        cursor.execute(f"UPDATE {table} SET lote = ? WHERE id = ?", (resultado.strip(), registro_id))

conn.commit()

Processando registro ID 2208
========== Verificando se é um caso de uso conhecido... ==========
Considere a seguinte lista de pedidos:
ID: 1, Resumo: Pedido de desistência do processo
ID: 2, Resumo: Informação de que a parte está ciente.
ID: 3, Resumo: Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança
ID: 4, Resumo: solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução de baixo valor
ID: 5, Resumo: Solicita a desconsideração do valor irrisório bloqueado e regular prosseguimento do processo
ID: 6, Resumo: Solicita a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas
É possível dizer que o pedido informado pode ser adequadamente descrito por um item dessa lista?.
Retorne APENAS o Número Identificador do item correspondente na lista, e mais nada.
Pedido:
ESTADO DO RIO GRANDE DO SUL
MUNICÍPIO DE SARANDI
EXCELENTÍSSIMA SENHORA DOUTORA JUÍZA DE DIREITO DA VARA
JUDIC

KeyboardInterrupt: 